In [1]:
import os
import cv2
import glob
import random
import shutil
import numpy as np

# ========== 原始資料路徑 ==========
SRC_BASE = r"C:\temp\yolo_FINAL"
SRC_IMG_BASE = os.path.join(SRC_BASE, "images")
SRC_LBL_BASE = os.path.join(SRC_BASE, "labels")

# ========== 擴增後儲存路徑 ==========
DST_BASE = r"C:\temp\yolo_FINAL_AUG"
DST_IMG_BASE = os.path.join(DST_BASE, "images")
DST_LBL_BASE = os.path.join(DST_BASE, "labels")

# ========== 建立資料夾 ==========
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(DST_IMG_BASE, split), exist_ok=True)
    os.makedirs(os.path.join(DST_LBL_BASE, split), exist_ok=True)

# ========== 工具函式 ==========
def rotate_image(img, angle):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1)
    return cv2.warpAffine(img, M, (w, h), borderValue=(0,0,0)), M

def rotate_bbox(cx, cy, M, img_w, img_h):
    abs_x = cx * img_w
    abs_y = cy * img_h
    new_x = M[0][0] * abs_x + M[0][1] * abs_y + M[0][2]
    new_y = M[1][0] * abs_x + M[1][1] * abs_y + M[1][2]
    return new_x / img_w, new_y / img_h

def add_noise(img):
    noise = np.random.normal(0, 10, img.shape).astype(np.uint8)
    return cv2.add(img, noise)

def apply_blur(img):
    return cv2.GaussianBlur(img, (5, 5), 0)

# ========== 資料擴增處理（僅 train） ==========
SRC_IMG_DIR = os.path.join(SRC_IMG_BASE, "train")
SRC_LBL_DIR = os.path.join(SRC_LBL_BASE, "train")
DST_IMG_DIR = os.path.join(DST_IMG_BASE, "train")
DST_LBL_DIR = os.path.join(DST_LBL_BASE, "train")

img_paths = glob.glob(os.path.join(SRC_IMG_DIR, "*.png"))
print("📂 找到 train 圖片數量：", len(img_paths))

for img_path in img_paths:
    base = os.path.splitext(os.path.basename(img_path))[0]
    label_path = os.path.join(SRC_LBL_DIR, base + ".txt")
    if not os.path.exists(label_path): continue

    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    # 讀取標註
    with open(label_path, 'r') as f:
        lines = f.read().splitlines()

    bboxes = []
    for line in lines:
        cls, cx, cy, bw, bh = map(float, line.strip().split())
        bboxes.append((int(cls), cx, cy, bw, bh))

    for i in range(3):  # 每張圖擴增3次
        aug_img = img.copy()
        aug_boxes = list(bboxes)

        if random.random() < 0.5:  # 水平翻轉
            aug_img = cv2.flip(aug_img, 1)
            aug_boxes = [(cls, 1 - cx, cy, bw, bh) for cls, cx, cy, bw, bh in aug_boxes]

        if random.random() < 0.9:  # ±20 度旋轉
            angle = random.uniform(-20, 20)
            aug_img, M = rotate_image(aug_img, angle)
            aug_boxes = [(cls, *rotate_bbox(cx, cy, M, w, h), bw, bh) for cls, cx, cy, bw, bh in aug_boxes]

        if random.random() < 0.5:  # 加噪
            aug_img = add_noise(aug_img)

        if random.random() < 0.5:  # 模糊
            aug_img = apply_blur(aug_img)

        # 儲存擴增圖片與標註
        new_img_name = f"{base}_aug{i}.png"
        new_lbl_name = f"{base}_aug{i}.txt"
        cv2.imwrite(os.path.join(DST_IMG_DIR, new_img_name), aug_img)
        with open(os.path.join(DST_LBL_DIR, new_lbl_name), 'w') as f:
            for cls, cx, cy, bw, bh in aug_boxes:
                f.write(f"{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")

print("✅ train 擴增完成！")

# ========== 複製 test 與 val ==========
for split in ['test', 'val']:
    src_img = os.path.join(SRC_IMG_BASE, split)
    src_lbl = os.path.join(SRC_LBL_BASE, split)
    dst_img = os.path.join(DST_IMG_BASE, split)
    dst_lbl = os.path.join(DST_LBL_BASE, split)

    for file in glob.glob(os.path.join(src_img, "*.png")):
        shutil.copy(file, dst_img)
    for file in glob.glob(os.path.join(src_lbl, "*.txt")):
        shutil.copy(file, dst_lbl)
    print(f"📋 已複製 {split} 資料 ({len(os.listdir(src_img))} 圖片)")

print("✅ 所有資料完成！")


📂 找到 train 圖片數量： 461
✅ train 擴增完成！
📋 已複製 test 資料 (142 圖片)
📋 已複製 val 資料 (115 圖片)
✅ 所有資料完成！
